# SOHO self-contained three-dataset study
This notebook evaluates the repository's current replay-based SOHO implementation with the same disciplined flow used by the SRQ study: train-only nested selection, immutable lock, six paired final replicates, tables, plots and a compact evidence ZIP. SOHO is **not exemplar-free**: its historical frozen-backbone features and labels are learner state and are counted in every state metric.

In [ ]:
# === Edit repository/path values only. Do not edit protocol seeds or search spaces. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'experiment/soho-selfcontained'
WORK_DIR = '/content/SOHO-CL'
FEATURE_CACHE_ROOT = '/content/soho_selfcontained_features'
SELECTION_ROOT = '/content/soho_selfcontained_selection'
OUTPUT_ROOT = '/content/soho_selfcontained_results'
BATCH_SIZE = 128
NUM_WORKERS = 2
EXPECTED_PROTOCOL_SHA256 = '6795af0056ec803c00eb83564821f9f8fecc10cae54d1e3b9297b7e725980a6e'
EXPECTED_RUNNER_SHA256 = 'ca774b9b68aee45d2ed1d3dcc8421060a4b3ad6d71472a7bead45e0fe97db0bc'

In [ ]:
# Fresh clone, dependency install, GPU check and immutable source verification.
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
os.chdir('/content')
repo = Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR], check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub','pandas','matplotlib','seaborn'], check=True)
import torch
assert torch.cuda.is_available(), 'Select Runtime -> Change runtime type -> T4 GPU.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
PROTOCOL = 'configs/soho_selfcontained_final.json'
RUNNER = 'tools/soho_selfcontained.py'
assert sha(PROTOCOL) == EXPECTED_PROTOCOL_SHA256
assert sha(RUNNER) == EXPECTED_RUNNER_SHA256
assert not subprocess.check_output(['git','status','--porcelain'], text=True).strip(), 'Repository must start clean.'
print('GPU:', torch.cuda.get_device_name(0))
print('commit:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())
print('protocol:', EXPECTED_PROTOCOL_SHA256)
print('runner:', EXPECTED_RUNNER_SHA256)

In [ ]:
# Download the verified frozen ViT checkpoint and processed datasets to temporary Colab storage.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH = hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size == 346284714
assert sha(CHECKPOINT_PATH) == '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
DATASET_ROOTS = {
  'cifar100': kagglehub.dataset_download('zaphat206/cifar-100'),
  'cub200': kagglehub.dataset_download('zaphat206/cub-200-2011'),
  'imagenetr': kagglehub.dataset_download('zaphat206/imagenet-r'),
}
print('checkpoint:', CHECKPOINT_PATH)
print(json.dumps(DATASET_ROOTS, indent=2))

In [ ]:
# Audit dataset identity without extracting test features.
CUB_AUDIT = '/content/cub_soho_audit.json'
IMAGENETR_AUDIT = '/content/imagenetr_soho_audit.json'
cub = subprocess.run([sys.executable,'-u','tools/cub_dataset_audit.py','--root',DATASET_ROOTS['cub200'],'--output',CUB_AUDIT,'--expected-identity-sha256','e374af9b576cb6b3503198ef3ea30fd0aa9d2e18c230ff8064e21d4f644af2ca'])
assert cub.returncode == 0
imagenetr = subprocess.run([sys.executable,'-u','tools/imagenetr_dataset_audit.py','--root',DATASET_ROOTS['imagenetr'],'--output',IMAGENETR_AUDIT,'--expected-identity-sha256','3f3d963b2b0c245ceabc0166c8b1c64d624c2ea31df07ee6ffdbf4cab5f7739d','--diagnose-cross-split-duplicates','--workers','4'])
assert imagenetr.returncode == 2, 'Expected locked legacy-overlap disclosure.'
audit = json.loads(Path(IMAGENETR_AUDIT).read_text())
assert audit['cross_split_duplicate_content_count'] == 19
assert audit['cross_split_conflicting_label_duplicate_count'] == 18
print('DATASET AUDIT PASS; ImageNet-R remains labeled legacy processed split.')

In [ ]:
# Extract frozen TRAIN features only, with one progress line per task. No Drive storage is used.
protocol = json.loads(Path(PROTOCOL).read_text())
Path(FEATURE_CACHE_ROOT).mkdir(parents=True, exist_ok=True)
for key in ('cifar100','cub200','imagenetr'):
    cfg = protocol['datasets'][key]; cache = Path(FEATURE_CACHE_ROOT)/key
    if not (cache/'train.pt').is_file():
        command = [sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only',
          '--root',DATASET_ROOTS[key],'--backbone-checkpoint',CHECKPOINT_PATH,
          '--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256',protocol['backbone']['checkpoint_sha256'],
          '--feature-cache-dir',str(cache),'--output-dir',f'/content/unused_{key}',
          '--dataset',cfg['dataset'],'--model-name',protocol['backbone']['model_name'],'--data-augmentation','vit',
          '--seed','2025','--num-classes',str(cfg['num_classes']),'--num-tasks',str(cfg['num_tasks']),
          '--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
        print(f'TRAIN EXTRACT START {key}', flush=True); subprocess.run(command, check=True)
    assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
    print(f'TRAIN CACHE READY {key}; test.pt absent')
print('ALL TRAIN-ONLY FEATURE CACHES READY')

In [ ]:
# Correctness and fidelity gate; synthetic data only.
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_soho_selfcontained.py','tests/test_cached_replay_baselines.py'], check=True)
print('SOHO SELF-CONTAINED CORRECTNESS GATE: PASS')

In [ ]:
# TRAIN-ONLY nested selection. Progress prints START, TASK and DONE lines.
audit_paths = {'cifar100': None, 'cub200': CUB_AUDIT, 'imagenetr': IMAGENETR_AUDIT}
for key in ('cifar100','cub200','imagenetr'):
    command = [sys.executable,'-u',RUNNER,'select','--protocol',PROTOCOL,'--dataset-key',key,
      '--feature-cache-dir',str(Path(FEATURE_CACHE_ROOT)/key),'--output-root',SELECTION_ROOT,'--device','cuda']
    if audit_paths[key]: command += ['--dataset-audit',audit_paths[key]]
    print(f'SELECTION START {key}: 11 sensitivity configs, then up to 6 local interactions; 3 development replicates', flush=True)
    subprocess.run(command, check=True)
    selected = json.loads(Path(SELECTION_ROOT,key,'selection.json').read_text())
    print(key, selected['status'], selected['selected_soho_config'], 'raw lambda=',selected['selected_raw_ridge_lambda'])
    assert selected['status'] == 'SELECTION_COMPLETE', 'Raw grid endpoint selected; stop before test and review.'
    assert selected['uses_test_set'] is False
print('TRAIN-ONLY SELECTION COMPLETE FOR ALL DATASETS')

In [ ]:
# Display validation evidence, plot coding/density sensitivity, then create the immutable lock.
import pandas as pd, matplotlib.pyplot as plt, numpy as np, seaborn as sns
selection_rows=[]; sensitivity_rows=[]; interaction_rows=[]
for key in ('cifar100','cub200','imagenetr'):
    result=json.loads(Path(SELECTION_ROOT,key,'selection.json').read_text())
    selection_rows.append({'dataset':key, **result['selected_soho_config'],
      'raw_ridge_lambda':result['selected_raw_ridge_lambda'],'status':result['status']})
    for candidate in result['stage1_sensitivity']:
      for replicate_index, replicate in enumerate(candidate['per_replicate']):
        sensitivity_rows.append({'dataset':key,'replicate':replicate_index,**candidate['config'],
          'validation_AIA':replicate['average_incremental_accuracy']})
    for candidate in result['stage2_interactions']:
      for replicate_index, replicate in enumerate(candidate['per_replicate']):
        interaction_rows.append({'dataset':key,'replicate':replicate_index,**candidate['config'],
          'validation_AIA':replicate['average_incremental_accuracy']})
display(pd.DataFrame(selection_rows))
sensitivity=pd.DataFrame(sensitivity_rows); interactions=pd.DataFrame(interaction_rows)
plot_dir=Path(OUTPUT_ROOT)/'validation_plots'; plot_dir.mkdir(parents=True,exist_ok=True)
sns.set_theme(style='whitegrid',context='talk')
fig,axes=plt.subplots(2,3,figsize=(22,11),sharey=False)
for column,dataset in enumerate(('cifar100','cub200','imagenetr')):
  coding=sensitivity[(sensitivity.dataset==dataset)&(sensitivity.density==0.3)].groupby('coding_level').validation_AIA.agg(['mean','std']).sort_index()
  axes[0,column].errorbar(coding.index,coding['mean'],yerr=coding['std'],marker='o',capsize=4,color='#E45756')
  axes[0,column].set_title(f'{dataset}: coding sweep at density=0.3'); axes[0,column].set_xlabel('Coding level'); axes[0,column].set_ylabel('Train-only validation AIA (%)')
  density=sensitivity[(sensitivity.dataset==dataset)&(sensitivity.coding_level==0.3)].groupby('density').validation_AIA.agg(['mean','std']).sort_index()
  axes[1,column].errorbar(density.index,density['mean'],yerr=density['std'],marker='o',capsize=4,color='#4C78A8')
  axes[1,column].set_title(f'{dataset}: density sweep at coding=0.3'); axes[1,column].set_xlabel('Projection density'); axes[1,column].set_ylabel('Train-only validation AIA (%)')
fig.tight_layout(); fig.savefig(plot_dir/'01_marginal_sensitivity.png',dpi=220,bbox_inches='tight'); plt.show()
fig,axes=plt.subplots(1,3,figsize=(22,6))
for ax,dataset in zip(axes,('cifar100','cub200','imagenetr')):
  table=interactions[interactions.dataset==dataset].groupby(['density','coding_level']).validation_AIA.mean().unstack()
  sns.heatmap(table,annot=True,fmt='.2f',cmap='viridis',ax=ax); ax.set_title(dataset); ax.set_xlabel('Coding level'); ax.set_ylabel('Projection density')
fig.tight_layout(); fig.savefig(plot_dir/'02_density_coding_heatmap.png',dpi=220,bbox_inches='tight'); plt.show()
display(sensitivity.groupby(['dataset','density','coding_level']).validation_AIA.agg(['mean','std']).reset_index())
display(interactions.groupby(['dataset','density','coding_level']).validation_AIA.agg(['mean','std']).reset_index())
dirty = subprocess.check_output(['git','status','--porcelain'], text=True).strip()
assert not dirty, f'Repository source changed before lock:\n{dirty}'
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable,'-u',RUNNER,'lock','--protocol',PROTOCOL,'--selection-root',SELECTION_ROOT,
  '--output-root',OUTPUT_ROOT,'--require-clean-git'], check=True)
AUTHORIZATION = str(Path(OUTPUT_ROOT)/'authorization.json')
print(json.dumps(json.loads(Path(AUTHORIZATION).read_text()), indent=2))

## Authorized test boundary
All grids, train-only selections, source hashes and seeds are now immutable. These test splits were used by earlier repository phases, so the results are a locked comparative evaluation rather than first-use untouched held-out evidence. Do not change any setting after viewing test output.

In [ ]:
# Materialize TEST features only after authorization.
for key in ('cifar100','cub200','imagenetr'):
    command = [sys.executable,'-u',RUNNER,'extract-test','--protocol',PROTOCOL,'--dataset-key',key,
      '--selection-root',SELECTION_ROOT,'--authorization',AUTHORIZATION,
      '--feature-cache-dir',str(Path(FEATURE_CACHE_ROOT)/key),'--root',DATASET_ROOTS[key],
      '--backbone-checkpoint',CHECKPOINT_PATH,'--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print(f'TEST EXTRACTION START {key}', flush=True); subprocess.run(command, check=True)
print('ALL AUTHORIZED TEST FEATURE CACHES READY')

In [ ]:
# Helper: refit every learner from empty state on full train, then evaluate six paired replicates.
def run_final(key):
    command=[sys.executable,'-u',RUNNER,'evaluate','--protocol',PROTOCOL,'--dataset-key',key,
      '--selection-root',SELECTION_ROOT,'--authorization',AUTHORIZATION,
      '--feature-cache-dir',str(Path(FEATURE_CACHE_ROOT)/key),'--output-root',OUTPUT_ROOT,'--device','cuda']
    if audit_paths[key]: command += ['--dataset-audit',audit_paths[key]]
    print(f'FINAL START {key}: 6 replicates x 3 methods', flush=True)
    subprocess.run(command, check=True)
    payload=json.loads(Path(OUTPUT_ROOT,key,'final_results.json').read_text())
    rows=[]
    for replicate in payload['seed_results']:
      for method,result in replicate['methods'].items():
        rows.append({'replicate':replicate['replicate_index'],'method':method,'status':result['status'],
          'final_accuracy':result.get('final_accuracy'),'AIA':result.get('average_incremental_accuracy'),
          'forgetting':result.get('forgetting'),'state_MiB':None if result.get('persistent_state_bytes') is None else result['persistent_state_bytes']/2**20,
          'exemplar_free':result.get('state_audit',{}).get('exemplar_free')})
    display(pd.DataFrame(rows)); print('STATUS:',payload['status'])

In [ ]:
# CIFAR-100 final evaluation.
run_final('cifar100')

In [ ]:
# CUB-200-2011 final evaluation.
run_final('cub200')

In [ ]:
# ImageNet-R legacy processed-split final evaluation.
run_final('imagenetr')

In [ ]:
# Aggregate means, sample standard deviations, 95% confidence intervals and paired differences.
subprocess.run([sys.executable,'-u',RUNNER,'summarize','--protocol',PROTOCOL,'--output-root',OUTPUT_ROOT], check=True)
metrics=pd.read_csv(Path(OUTPUT_ROOT)/'metrics_summary.csv')
display(metrics.sort_values(['dataset','method']))
summary=json.loads(Path(OUTPUT_ROOT,'final_summary.json').read_text())
print('Paired AIA differences:',json.dumps(summary['paired_aia_differences'],indent=2))
print('SOHO exemplar-free:',summary['soho_exemplar_free'])
print('Disclosure:',summary['test_reuse_disclosure'])

In [ ]:
# Publication-style plots: curves, accuracy, state, runtime, Pareto and paired SOHO-FLY differences.
import matplotlib.pyplot as plt, numpy as np, seaborn as sns
sns.set_theme(style='whitegrid',context='talk')
plot_dir=Path(OUTPUT_ROOT)/'plots'; plot_dir.mkdir(parents=True,exist_ok=True)
curves=pd.read_csv(Path(OUTPUT_ROOT)/'task_curves.csv')
names={'soho_replay_fidelity':'SOHO replay','flycl_fidelity':'FLY-CL','raw_ridge':'Raw Ridge'}
colors={'soho_replay_fidelity':'#E45756','flycl_fidelity':'#4C78A8','raw_ridge':'#54A24B'}
datasets=['cifar100','cub200','imagenetr']; methods=list(names)
fig,axes=plt.subplots(1,3,figsize=(21,5.5),sharey=True)
for ax,dataset in zip(axes,datasets):
  view=curves[curves.dataset==dataset]
  for method in methods:
    group=view[view.method==method].groupby('task').average_seen_accuracy
    mean=group.mean(); ci=2.571*group.std(ddof=1)/np.sqrt(group.count())
    ax.plot(mean.index,mean.values,label=names[method],color=colors[method],marker='o',ms=3)
    ax.fill_between(mean.index,mean-ci,mean+ci,color=colors[method],alpha=.16)
  ax.set_title(dataset); ax.set_xlabel('Task'); ax.set_ylabel('Average seen-class accuracy (%)')
axes[0].legend(fontsize=10); fig.tight_layout(); fig.savefig(plot_dir/'01_accuracy_by_task.png',dpi=220,bbox_inches='tight'); plt.show()
fig,axes=plt.subplots(1,2,figsize=(15,5.5))
for ax,metric,title in zip(axes,['final_accuracy','average_incremental_accuracy'],['Final accuracy','Average incremental accuracy']):
  x=np.arange(3); width=.25
  for j,method in enumerate(methods):
    view=metrics[metrics.method==method].set_index('dataset').loc[datasets]
    y=view[f'{metric}_mean'].to_numpy(); err=np.vstack([y-view[f'{metric}_ci95_low'],view[f'{metric}_ci95_high']-y])
    ax.bar(x+(j-1)*width,y,width,label=names[method],color=colors[method],yerr=err,capsize=4)
  ax.set_xticks(x,datasets); ax.set_ylabel('Accuracy (%)'); ax.set_title(title)
axes[0].legend(fontsize=10); fig.tight_layout(); fig.savefig(plot_dir/'02_final_and_aia.png',dpi=220,bbox_inches='tight'); plt.show()
for metric,title,filename,logscale in [('persistent_state_bytes','Persistent learner state (MiB)','03_persistent_state.png',True),('peak_runtime_memory_bytes','Peak allocated GPU memory (MiB)','04_peak_runtime_memory.png',False),('total_update_seconds','Total analytic update time (s)','05_update_time.png',False)]:
  fig,ax=plt.subplots(figsize=(10,5.5)); x=np.arange(3); width=.25
  for j,method in enumerate(methods):
    view=metrics[metrics.method==method].set_index('dataset').loc[datasets]; values=view[f'{metric}_mean'].to_numpy()/(2**20 if 'bytes' in metric else 1)
    ax.bar(x+(j-1)*width,values,width,label=names[method],color=colors[method])
  if logscale: ax.set_yscale('log')
  ax.set_xticks(x,datasets); ax.set_ylabel(title); ax.legend(fontsize=10); fig.tight_layout(); fig.savefig(plot_dir/filename,dpi=220,bbox_inches='tight'); plt.show()
fig,axes=plt.subplots(1,3,figsize=(18,5.5),sharey=True)
for ax,dataset in zip(axes,datasets):
  view=metrics[metrics.dataset==dataset]
  for method in methods:
    row=view[view.method==method].iloc[0]; x=row.persistent_state_bytes_mean/2**20; y=row.average_incremental_accuracy_mean
    ax.scatter(x,y,s=130,color=colors[method]); ax.annotate(names[method],(x,y),xytext=(5,5),textcoords='offset points',fontsize=9)
  ax.set_xscale('log'); ax.set_title(dataset); ax.set_xlabel('Persistent state (MiB, log)'); ax.set_ylabel('AIA (%)')
fig.tight_layout(); fig.savefig(plot_dir/'06_accuracy_memory_pareto.png',dpi=220,bbox_inches='tight'); plt.show()
paired_rows=[]
for dataset in datasets:
  payload=json.loads(Path(OUTPUT_ROOT,dataset,'final_results.json').read_text())
  for item in payload['seed_results']:
    paired_rows.append({'dataset':dataset,'replicate':item['replicate_index'],'SOHO minus FLY AIA':item['methods']['soho_replay_fidelity']['average_incremental_accuracy']-item['methods']['flycl_fidelity']['average_incremental_accuracy']})
paired_df=pd.DataFrame(paired_rows); fig,ax=plt.subplots(figsize=(10,5.5)); sns.stripplot(data=paired_df,x='dataset',y='SOHO minus FLY AIA',jitter=False,size=9,ax=ax)
ax.axhline(0,color='black',lw=1); ax.set_ylabel('Paired SOHO - FLY AIA (pp)')
fig.tight_layout(); fig.savefig(plot_dir/'07_paired_soho_fly.png',dpi=220,bbox_inches='tight'); plt.show()
print('PLOTS:',sorted(path.name for path in plot_dir.glob('*.png')))

In [ ]:
# Export compact evidence. Feature caches and SOHO replay tensors are deliberately excluded from the ZIP.
from google.colab import files
shutil.copy2(PROTOCOL,Path(OUTPUT_ROOT)/'locked_protocol.json')
shutil.copy2(RUNNER,Path(OUTPUT_ROOT)/'locked_runner.py')
selection_copy=Path(OUTPUT_ROOT)/'train_only_selection'; selection_copy.mkdir(exist_ok=True)
for key in ('cifar100','cub200','imagenetr'): shutil.copy2(Path(SELECTION_ROOT,key,'selection.json'),selection_copy/f'{key}_selection.json')
archive=shutil.make_archive('/content/soho_selfcontained_three_dataset_results','zip',root_dir=OUTPUT_ROOT)
print('artifact:',archive,'size=',Path(archive).stat().st_size,'SHA-256=',sha(archive))
files.download(archive)